# CAPEN-Llama attention γ on TopTagging $k$NN

Estimate pre-softmax temperatures $\gamma_{rs}$ as **mean nonempty row degrees** of supports $M_{11}/M_{12}/M_{21}/M_{22}$ on the directed $k$NN train cache.

For **CAPEN-Llama**, $M_{11}$ is all-to-all among real particles (so $\gamma_{11}\approx \langle N\rangle$).

Default cache: `processed/4nn/n128/` (plain $4$-NN, no $p_T$ cut).

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Login nodes: avoid CUDA probe hangs.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("OMP_NUM_THREADS", "4")

import matplotlib.pyplot as plt
import numpy as np

REPO = Path("..").resolve()
for p in (REPO, REPO / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from cpen.utils.attention_temperature import estimate_attention_gammas_from_cache
from cpen.utils.toptagging import normalize_data_root

DATA_ROOT = normalize_data_root("/scratch/gpfs/BHANIN/jdezoort/datasets/toptagging")
GRAPH_CONSTRUCTION = "4-NN"
NUM_PARTICLES = 128
N_JETS = 512
SEED = 42

# CAPEN-Llama: all-to-all particle attention for M_11
ALL_TO_ALL_11 = True

print(f"DATA_ROOT={DATA_ROOT}")
print(f"graph={GRAPH_CONSTRUCTION}  n={NUM_PARTICLES}  n_jets={N_JETS}")

In [ ]:
gammas, summaries = estimate_attention_gammas_from_cache(
    data_root=str(DATA_ROOT),
    split="train",
    graph_construction=GRAPH_CONSTRUCTION,
    num_particles=NUM_PARTICLES,
    n_jets=N_JETS,
    seed=SEED,
    all_to_all_particle_attention=ALL_TO_ALL_11,
)

print("Recommended --attention-normalization gamma values (mean row degrees):")
print(
    f"  --gamma-11 {gammas.gamma_11:.4g} "
    f"--gamma-12 {gammas.gamma_12:.4g} "
    f"--gamma-21 {gammas.gamma_21:.4g} "
    f"--gamma-22 {gammas.gamma_22:.4g}"
)
print()
for rel, s in summaries.items():
    print(
        f"M_{rel}: n_rows={s['n_rows']:,}  "
        f"mean={s['mean']:.3f}  median={s['median']:.3f}  "
        f"std={s['std']:.3f}  min={s['min']:.0f}  max={s['max']:.0f}"
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharey=False)
for ax, rel in zip(axes.ravel(), ("11", "12", "21", "22")):
    vals = summaries[rel]["degrees"]
    ax.hist(
        vals,
        bins=min(40, max(5, int(np.sqrt(max(len(vals), 1))))),
        color="#4C72B0",
        edgecolor="white",
    )
    ax.axvline(
        summaries[rel]["mean"],
        color="#C44E52",
        ls="--",
        lw=1.5,
        label=rf"$\gamma$={summaries[rel]['mean']:.2f}",
    )
    ax.set_title(rf"$M_{{{rel}}}$ row degrees")
    ax.set_xlabel(r"$n_i$")
    ax.legend(frameon=False)
axes[0, 0].set_ylabel("count")
axes[1, 0].set_ylabel("count")
fig.suptitle(
    f"{GRAPH_CONSTRUCTION}, N={NUM_PARTICLES}, "
    f"all-to-all M₁₁={ALL_TO_ALL_11}, n_jets={N_JETS}",
    y=1.02,
)
fig.tight_layout()
plt.show()

## Paste into training

```bash
sbatch --export=ALL,KNN_K=4 scans/testing/test_capen_llama_toptagging_kNN.slurm
```

Or set the printed `--gamma-11/12/21/22` in that slurm script (defaults should match this notebook once you update them).

Auto-estimate at train time:

```bash
python scans/testing/test_cpen_toptagging.py \\
  --model capen-llama --graph-construction 4-NN \\
  --attention-normalization gamma \\
  --estimate-gamma-rs --gamma-estimate-jets 512 ...
```